# 01 · Auditoría de datos: v0.2 vs v3

**Objetivo.** Medir con los datos reales qué cambió entre el ETL de la v0.2 y el de la v3, y separar cada efecto:

1. **Estado de calidad v3**: qué marcó la validación y por qué.
2. **Recorte (winsorización) de v0.2**: cuánto subestimaba la cola extrema.
3. **Dividendos**: cuánto retorno perdía v0.2 al usar el cierre sin ajustar.
4. **Portafolio**: por qué el VaR al 5 % casi no cambió (datos, calendario y ventana).
5. **Días marcados**: revisión de los flags, incluido el detalle de ETH.

**Entradas**
- `data/legacy_v02/raw/*.csv`: datos v0.2 (Yahoo Finance, recortados).
- `data/processed/*.csv`: datos v3 (Tiingo y Coinbase, validados).
- `artifacts/data_quality/latest.json`: informe del último ETL.

**Requisito:** haber corrido `python -m alphaquant.data.pipeline`.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from alphaquant.core import paths
from alphaquant.core.config import load_config
from alphaquant.data.calendars import aggregate_crypto_to_equity_sessions, expected_dates, is_crypto
from alphaquant.risk.portfolio_analytics import (
    historical_cvar,
    historical_var,
    max_drawdown,
    portfolio_returns,
    sharpe_ratio,
)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_rows", 80)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})

LEGACY_DIR = paths.data_dir() / "legacy_v02" / "raw"
PROCESSED_DIR = paths.processed_dir()
REPORT_FILE = paths.artifacts_dir() / "data_quality" / "latest.json"
TICKERS = load_config()["tickers"]

for p in (LEGACY_DIR, PROCESSED_DIR, REPORT_FILE):
    assert p.exists(), f"Falta {p}: corre primero el ETL (y la migracion v3 para legacy_v02)"
print("Raiz del proyecto:", paths.project_root())
print("Tickers:", len(TICKERS))

## Carga de datos

- **v0.2**: `ret` es el retorno log del cierre **sin ajustar por dividendos** y **recortado** a los cuantiles 0.1 %/99.9 % de toda la muestra.
- **v3**: `ret` es el retorno log **total** (desde `adj_close`, con dividendos) y **sin recortes**.

Para aislar efectos se calcula además `ret_price` en v3: el retorno de precio ajustado solo por splits, sin dividendos. Es el equivalente "limpio" de lo que v0.2 intentaba medir.

In [ ]:
def load_legacy(ticker: str) -> pd.DataFrame:
    df = pd.read_csv(LEGACY_DIR / f"{ticker}_1d.csv", usecols=["Datetime", "Close", "AdjClose", "ret"])
    df["date"] = pd.to_datetime(df["Datetime"], utc=True).dt.tz_localize(None).dt.normalize()
    df = df.drop_duplicates("date").set_index("date").sort_index()
    return df.rename(columns={"Close": "close", "AdjClose": "adj_close"})[["close", "adj_close", "ret"]]


def load_v3(ticker: str) -> pd.DataFrame:
    df = pd.read_csv(PROCESSED_DIR / f"{ticker}_1d.csv", parse_dates=["date"]).set_index("date")
    # F_t = producto de los splits POSTERIORES a t: lleva el precio crudo a la base actual.
    future_splits = df["split_factor"][::-1].cumprod()[::-1].shift(-1).fillna(1.0)
    df["ret_price"] = np.log(df["close"] / future_splits).diff()
    return df


legacy = {t: load_legacy(t) for t in TICKERS}
v3 = {t: load_v3(t) for t in TICKERS}

pd.DataFrame({
    t: {"v0.2 desde": legacy[t].index.min().date(), "v0.2 hasta": legacy[t].index.max().date(),
        "v0.2 filas": len(legacy[t]), "v3 desde": v3[t].index.min().date(),
        "v3 hasta": v3[t].index.max().date(), "v3 filas": len(v3[t])}
    for t in TICKERS
}).T

## 1 · Estado de calidad v3

Resumen del informe del ETL. Un **WARN** no es un error: significa que hay días marcados para revisar. Un ticker en **FAIL** no se publica.

In [ ]:
report = json.loads(REPORT_FILE.read_text(encoding="utf-8"))
print(f"Corrida {report['run_id']} | estado global: {report['overall']}")

rows = []
for t in report["tickers"]:
    st = t.get("stats") or {}
    flags = {k.replace("flag_", ""): v for k, v in (st.get("flags") or {}).items()}
    rows.append({"ticker": t["ticker"], "fuente": t["provider"], "estado": t["status"],
                 "desde": st.get("first_date"), "faltantes": st.get("missing_dates"), **flags})
quality = pd.DataFrame(rows).set_index("ticker")
quality

## 2 · Efecto del recorte de v0.2

Se compara `ret` de v0.2 contra `ret_price` de v3, **en las mismas fechas**, para medir solo el efecto del recorte.

Dos advertencias para leer la tabla:
- Las fuentes son distintas (Yahoo vs Tiingo/Coinbase). Por eso hay pequeñas diferencias diarias que no vienen del recorte. Se cuentan como "días distintos" las diferencias mayores a 0.1 %.
- En cripto, la diferencia mezcla el recorte con el cambio de fuente: Yahoo agrega varios exchanges y Coinbase es uno solo.

**Qué esperar:** el recorte afectaba al 0.1 % de los días más extremos de cada lado, así que debe notarse en el **peor día** y en el **VaR/CVaR al 1 %**, y casi nada en el VaR al 5 %.

In [ ]:
def tail(r: pd.Series) -> dict:
    return {"peor": r.min(), "VaR5": historical_var(r, 0.05), "CVaR5": historical_cvar(r, 0.05),
            "VaR1": historical_var(r, 0.01), "CVaR1": historical_cvar(r, 0.01)}


rows = []
for t in TICKERS:
    old = legacy[t]["ret"].dropna()
    new = v3[t]["ret_price"].dropna()
    idx = old.index.intersection(new.index)
    o, n = tail(old.loc[idx]), tail(new.loc[idx])
    rows.append({
        "ticker": t, "dias": len(idx),
        "dias_distintos": int(((old.loc[idx] - new.loc[idx]).abs() > 1e-3).sum()),
        "peor_v02": o["peor"], "peor_v3": n["peor"],
        "VaR5_v02": o["VaR5"], "VaR5_v3": n["VaR5"],
        "CVaR1_v02": o["CVaR1"], "CVaR1_v3": n["CVaR1"],
        "subestim_CVaR1_%": 100 * (1 - o["CVaR1"] / n["CVaR1"]),
        "subestim_VaR5_%": 100 * (1 - o["VaR5"] / n["VaR5"]),
    })
clipping = pd.DataFrame(rows).set_index("ticker")
clipping

In [ ]:
print(f"Subestimacion promedio del CVaR 1 %: {clipping['subestim_CVaR1_%'].mean():.1f} %")
print(f"Subestimacion promedio del VaR 5 %:  {clipping['subestim_VaR5_%'].mean():.1f} %")

ax = clipping[["peor_v02", "peor_v3"]].mul(100).plot.bar(figsize=(12, 4), color=["#bbbbbb", "#c0392b"])
ax.set_title("Peor dia por ticker: v0.2 (recortado) vs v3 (real)")
ax.set_ylabel("retorno log diario (%)")
ax.legend(["v0.2", "v3"])
plt.show()

In [ ]:
window = slice("2020-02-20", "2020-04-15")
spy = pd.DataFrame({"v0.2": legacy["SPY"]["ret"], "v3": v3["SPY"]["ret_price"]}).loc[window].mul(100)

ax = spy.plot(marker="o", ms=3, color=["#bbbbbb", "#c0392b"])
ax.axhline(0, color="black", lw=0.8)
ax.set_title("SPY, crash del COVID: los dias extremos que v0.2 recortaba")
ax.set_ylabel("retorno log diario (%)")
plt.show()
spy[(spy["v0.2"] - spy["v3"]).abs() > 0.1]

## 3 · Efecto de los dividendos

v0.2 calculaba el retorno con el cierre sin ajustar: el día del pago, el precio cae por el monto del dividendo y eso se registraba como pérdida. v3 usa el precio ajustado, es decir, el retorno total.

La diferencia anual entre `ret` (total) y `ret_price` (solo precio) es el rendimiento por dividendos que v0.2 no contaba. Las cripto no pagan dividendos, así que su diferencia debe ser 0.

In [ ]:
rows = []
for t in TICKERS:
    d = v3[t].dropna(subset=["ret", "ret_price"])
    ppy = 365 if is_crypto(t) else 252
    rows.append({"ticker": t,
                 "retorno_total_anual_%": 100 * d["ret"].mean() * ppy,
                 "retorno_precio_anual_%": 100 * d["ret_price"].mean() * ppy,
                 "dividendos_anual_pp": 100 * (d["ret"] - d["ret_price"]).mean() * ppy,
                 "pagos": int((d["dividend"] > 0).sum())})
dividends = pd.DataFrame(rows).set_index("ticker").sort_values("dividendos_anual_pp", ascending=False)
dividends

In [ ]:
hyg = v3["HYG"].dropna(subset=["ret", "ret_price"])
growth = pd.DataFrame({"precio (v0.2)": np.exp(hyg["ret_price"].cumsum()),
                       "retorno total (v3)": np.exp(hyg["ret"].cumsum())})
ax = growth.plot(color=["#bbbbbb", "#2e86c1"])
ax.set_title("HYG: crecimiento de 1 USD, solo precio vs retorno total")
plt.show()

## 4 · Portafolio: por qué el VaR al 5 % casi no cambió

En el portafolio cambiaron tres cosas a la vez. Para separarlas, se calcula el mismo portafolio (pesos iguales) en cuatro escenarios, cambiando una cosa por vez:

| Escenario | Datos | Calendario cripto | Ventana |
|---|---|---|---|
| A | v0.2 | se descartan fines de semana | la de v0.2 |
| B | v3 | se descartan fines de semana | la de v0.2 |
| C | v3 | fin de semana sumado al lunes | la de v0.2 |
| D | v3 | fin de semana sumado al lunes | completa (lo que reporta el pipeline) |

- A → B: efecto de los datos (recorte, dividendos y fuente).
- B → C: efecto del calendario.
- C → D: efecto de la ventana. Coinbase tiene ETH desde 2016 y Yahoo desde finales de 2017.

In [ ]:
def matrix(source: dict, col: str) -> pd.DataFrame:
    return pd.DataFrame({t: source[t][col] for t in TICKERS})


def summary(r: pd.Series) -> dict:
    return {"desde": r.index.min().date(), "hasta": r.index.max().date(), "dias": len(r),
            "VaR5": historical_var(r, 0.05), "CVaR5": historical_cvar(r, 0.05),
            "VaR1": historical_var(r, 0.01), "CVaR1": historical_cvar(r, 0.01),
            "max_dd": max_drawdown(r), "sharpe": sharpe_ratio(r)}


port_a = portfolio_returns(matrix(legacy, "ret"))
lo, hi = port_a.index.min(), port_a.index.max()

v3_ret = matrix(v3, "ret")
port_b = portfolio_returns(v3_ret.loc[lo:hi])
port_c = portfolio_returns(aggregate_crypto_to_equity_sessions(v3_ret).loc[lo:hi])
port_d = portfolio_returns(aggregate_crypto_to_equity_sessions(v3_ret))

scenarios = pd.DataFrame({"A v0.2": summary(port_a), "B v3 datos": summary(port_b),
                          "C + calendario": summary(port_c), "D + ventana": summary(port_d)}).T
scenarios

**Cómo leer la tabla.** Mira primero las columnas `VaR1` y `CVaR1` entre A y B: ahí aparece el recorte. Después mira `VaR5` entre C y D, que muestra el efecto de la ventana.

El `sharpe` de B en adelante incluye dividendos. La fila D debe coincidir con `artifacts/risk/portfolio_summary.csv`; si no coincide, hay que revisar el pipeline.

## 5 · Días marcados

Todos los días con algún flag, para revisarlos uno por uno. Un outlier confirmado en las noticias (crash del COVID, resultados trimestrales) es un **evento real** y se conserva. Un error del proveedor se vería como un salto que no aparece en ninguna otra fuente.

In [ ]:
flag_cols = [c for c in next(iter(v3.values())).columns if c.startswith("flag_") and c != "flag_any"]
flagged = []
for t in TICKERS:
    d = v3[t]
    hit = d[d["flag_any"]]
    for date, row in hit.iterrows():
        flagged.append({"ticker": t, "fecha": date.date(), "ret_%": 100 * row["ret"],
                        "flags": ", ".join(c.replace("flag_", "") for c in flag_cols if row[c])})
flagged = pd.DataFrame(flagged).sort_values(["fecha", "ticker"])
print(f"{len(flagged)} dias marcados en {flagged['ticker'].nunique()} tickers")
flagged.groupby("flags").size().rename("dias").to_frame()

In [ ]:
by_date = flagged.groupby("fecha")["ticker"].count().sort_values(ascending=False).head(10)
print("Fechas con mas tickers marcados a la vez (eventos de mercado, no errores de un proveedor):")
by_date.to_frame("tickers_marcados")

### Detalle de ETH-USD

ETH tiene precios congelados (`stale`) y fechas faltantes, ambos de los primeros años en Coinbase, con poco volumen. Aquí se ve cada caso con su volumen, para decidir si se acepta o se excluye ese tramo de la historia.

In [ ]:
eth = v3["ETH-USD"]
stale_dates = eth.index[eth["flag_stale"]]
context = sorted({d + pd.Timedelta(days=k) for d in stale_dates for k in (-2, -1, 0, 1)} & set(eth.index))
display(eth.loc[context, ["open", "high", "low", "close", "volume", "ret", "flag_stale"]])

expected = expected_dates("ETH-USD", eth.index.min(), eth.index.max())
missing = expected.difference(eth.index)
print("Fechas faltantes en ETH-USD:", [d.date() for d in missing])

## Conclusiones

Completa esta sección con los números de las tablas de arriba. Así queda escrita con tus palabras, que es lo que vas a tener que defender.

1. **Recorte de v0.2:** ¿cuánto subestimaba el CVaR al 1 % en promedio y en el peor caso? ¿Por qué el VaR al 5 % casi no cambia?
2. **Dividendos:** ¿qué activos perdían más rendimiento y cuánto al año?
3. **Portafolio:** de la diferencia entre A y D, ¿cuánto se explica por los datos, cuánto por el calendario y cuánto por la ventana?
4. **Flags:** ¿hay algún día marcado que no corresponda a un evento real conocido? ¿Se acepta el tramo inicial de ETH?

**Siguiente paso:** contrastar los días marcados con una segunda fuente para separar automáticamente los eventos confirmados de los posibles errores del proveedor.